In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
conn = sqlite3.connect("base_tratada.db")

query = """
SELECT *
FROM base_tratada;
"""

df = pd.read_sql(query, conn)

# Visualizar os dados
df.head()

#testando se o banco conectou

In [ ]:
df_tratado = df[df["preco"] <= 7000].copy()

#Limita o preco das diaria ate 7k

In [ ]:
df_tratado_price_min = df_tratado[
    (df_tratado["preco"] >= 65) &
    (df_tratado["quartos"] <= 15) &
    (df_tratado["banheiros"] <= 12)
].copy()

#atribui ao novo dataframe valores de limete pós remocao de outliers

In [ ]:
df_tratado_price_min[
    [
        "preco",
        "hospedes",
        "quartos",
        "camas",
        "nota_avaliacao",
        "noites_minimas",
        "quantidade_avaliacoes",
        "latitude",
        "longitude",
        "banheiros",
        "tipo_de_quarto"
    ]
].describe()
df_tratado_price_min.info()

In [ ]:
variaveis = [
    "preco",
    "quartos",
    "banheiros",
    "latitude",
    "longitude",
    "camas",
    "nota_avaliacao",
    "quantidade_avaliacoes",
    "hospedes",
    "noites_minimas",
    "tipo_quarto_casa_apto_inteiro",
    "tipo_quarto_quarto_compartilhado",
    "tipo_quarto_quarto_hotel",
    "tipo_quarto_quarto_privativo",
    "tipo_de_propriedade_grupo",
    "ipca_mensal",
    "ipca_acumulado_ano"
]

df_corr = df_tratado_price_min[variaveis]

In [ ]:
corr_matrix = df_corr.corr()

In [ ]:
plt.figure(figsize=(12, 9))

sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    linewidths=0.5
)

plt.title("Matriz de Correlação das Variáveis com o Preço da Diária")
plt.show()

Treinando Modelo de Regressão Linear

In [ ]:
# variável alvo
y = df_tratado_price_min["preco"]

# variáveis explicativas
X = df_tratado_price_min.drop(columns=["preco", "ano_mes", "bairro_encode", "bairro", "tipo_de_propriedade", "tipo_de_quarto", "ano", "mes", "ipca_acumulado_ano", "ipca_mensal"])
X.head()


In [ ]:

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

#train/test split, separa em dados para treino e para teste

In [ ]:
from sklearn.impute import SimpleImputer

imputador = SimpleImputer(strategy="median")
X_train = imputador.fit_transform(X_train)
X_test = imputador.transform(X_test)

In [ ]:
from sklearn.linear_model import LinearRegression

modelo = LinearRegression(
)

modelo.fit(X_train, y_train)
#treina o modelo pela primeira vez

In [ ]:
y_pred = modelo.predict(X_test)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae_full = mean_absolute_error(y_test, y_pred)
rmse_full = np.sqrt(mean_squared_error(y_test, y_pred))
r2_full = r2_score(y_test, y_pred)

In [ ]:
y_pred

Resultados

In [ ]:
print("MODELO SEM LOG")
print("MAE:", mae_full)
print("RMSE:", rmse_full)
print("R²:", r2_full)

In [ ]:
import pandas as pd

coeficientes = pd.Series(
    modelo.coef_,
    index=X.columns
).sort_values(key=abs, ascending=False)

print(coeficientes)



In [ ]:

coeficientes.head(10).plot(kind="barh")
plt.title("Coeficientes mais importantes (Regressão Linear)")
plt.gca().invert_yaxis()
plt.show()


Treinando Modelo de Regressão Linear Com Transformação de Log

In [ ]:

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)


In [ ]:

from sklearn.impute import SimpleImputer

imputador = SimpleImputer(strategy="median")
X_train = imputador.fit_transform(X_train)
X_test = imputador.transform(X_test)


In [ ]:


from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [ ]:

import numpy as np

y_train_log = np.log1p(y_train)


In [ ]:

from sklearn.linear_model import LinearRegression

modelo = LinearRegression()
modelo.fit(X_train, y_train_log)


In [ ]:
y_pred_log = modelo.predict(X_test)

In [ ]:
y_pred = np.expm1(y_pred_log)
y_pred = np.maximum(0, y_pred)


In [ ]:
y_pred

Resultados

In [ ]:

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("REGRESSÃO LINEAR COM TRANSFORMAÇÃO DE LOG")
print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)


In [ ]:

import pandas as pd

coeficientes = pd.Series(
    modelo.coef_,
    index=X.columns
).sort_values(key=abs, ascending=False)

print(coeficientes)
